# NLP

## Setup 

This setup allows you to use *Python* and *R* in the same notebook.


In [1]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [2]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [3]:
%%R

# My commonly used R imports

require('tidyverse')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: tidyverse


In [4]:
from tqdm.notebook import tqdm
tqdm.pandas()

## Load Data & Remove Duplciates 🧹

In [5]:
# read data
stories = pd.read_csv("output/stories_df.csv", 
                 parse_dates=['publication_date', 'capture_time'])


duplicates to delete

In [6]:
dedupe_by = ['title', 'domain']
stories[stories.duplicated(subset=dedupe_by, keep=False)]\
    .sort_values(by=dedupe_by)\
    .head()

,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet
26,2024 Election Live Updates: Latest Harris and ...,2024-09-23,2024-09-24 01:09:17+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/23/us/tru...,https://web.archive.org/web/20240924010917id_/...,https://web.archive.org/web/20240924010917/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Trump Hosts Rally in Pe...
1223,2024 Election Live Updates: Latest Harris and ...,2024-09-14,2024-09-15 01:39:39+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/14/us/har...,https://web.archive.org/web/20240915013939id_/...,https://web.archive.org/web/20240915013939/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Harris and Trump Keep U...
3246,"ABBA calls out Trump for ""unauthorized use"" of...",2024-08-30,2024-08-31 02:06:22+00:00,en,cbsnews.com,https://www.cbsnews.com/news/abba-trump-campai...,https://web.archive.org/web/20240831020622id_/...,https://web.archive.org/web/20240831020622/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"ABBA calls out Trump for ""unauthorized use"" of..."
3362,"ABBA calls out Trump for ""unauthorized use"" of...",2024-08-30,2024-09-01 01:41:57+00:00,en,cbsnews.com,https://www.cbsnews.com/colorado/news/abba-tru...,https://web.archive.org/web/20240901014157id_/...,https://web.archive.org/web/20240901014157/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"ABBA calls out Trump for ""unauthorized use"" of..."
3448,Army says Arlington National Cemetery official...,2024-08-29,2024-08-30 01:58:26+00:00,en,cbsnews.com,https://www.cbsnews.com/chicago/news/arlington...,https://web.archive.org/web/20240830015826id_/...,https://web.archive.org/web/20240830015826/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Army says Arlington National Cemetery official...


In [7]:
# remove duplicates
stories.drop_duplicates(subset=dedupe_by, keep='last', inplace=True)

# preview
stories

,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet
0,Man suspected of assassination attempt against...,2024-09-23,2024-09-24 01:06:28+00:00,en,cnn.com,https://www.cnn.com/2024/09/23/politics/ryan-w...,https://web.archive.org/web/20240924010628id_/...,https://web.archive.org/web/20240924010628/htt...,https://wayback-api.archive.org/colsearch/v1/m...,The man who authorities say sat with a rifle i...
1,Kamala Harris' presidential betting lead grows...,2024-09-23,2024-09-24 01:09:20+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/e...,https://web.archive.org/web/20240924010920id_/...,https://web.archive.org/web/20240924010920/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Kamala Harris pulls further ahead in president...
2,RFK Jr. asks Supreme Court to keep his name on...,2024-09-23,2024-09-24 01:15:44+00:00,en,cnn.com,https://www.cnn.com/2024/09/23/politics/rfk-jr...,https://web.archive.org/web/20240924011544id_/...,https://web.archive.org/web/20240924011544/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Robert F. Kennedy Jr. asked the Supreme Court ...
3,Nebraska change that could hand Donald Trump w...,2024-09-23,2024-09-24 01:12:31+00:00,en,newsweek.com,https://www.newsweek.com/nebraska-change-votes...,https://web.archive.org/web/20240924011231id_/...,https://web.archive.org/web/20240924011231/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Republican South Carolina Senator Lindsey Grah...
4,Judge detains Ryan Routh in Trump stakeout 'as...,2024-09-23,2024-09-24 01:10:27+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/2...,https://web.archive.org/web/20240924011027id_/...,https://web.archive.org/web/20240924011027/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Judge detains Ryan Routh after prosecutors say...
...,...,...,...,...,...,...,...,...,...,...
4230,Kamala Harris’s Main-Character Energy,2024-08-23,2024-08-24 01:58:41+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/arts/televi...,https://web.archive.org/web/20240824015841id_/...,https://web.archive.org/web/20240824015841/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Supported by\nCritic’s Notebook\nKamala Harris...
4231,"‘Joy,’ ‘Freedom,’ ‘Goldilocks’: Kamala Harris’...",2024-08-23,2024-08-24 02:08:37+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/books/revie...,https://web.archive.org/web/20240824020837id_/...,https://web.archive.org/web/20240824020837/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"Supported by\nCritic’s Notebook\n‘Joy,’ ‘Freed..."
4232,Why Kamala Harris’s Centrism Is Working,2024-08-23,2024-08-24 01:52:03+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/briefing/ka...,https://web.archive.org/web/20240824015203id_/...,https://web.archive.org/web/20240824015203/htt...,https://wayback-api.archive.org/colsearch/v1/m...,The Morning\nKamala Harris capped her first mo...
4233,Kamala Harris Wants to Win,2024-08-23,2024-08-24 02:11:07+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/opinion/ezr...,https://web.archive.org/web/20240824021107id_/...,https://web.archive.org/web/20240824021107/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"On Thursday night, Kamala Harris reintroduced ..."


# Keywords

In [8]:
from yake import KeywordExtractor
from pandarallel import pandarallel

kw_extractor = KeywordExtractor()

def get_keywords(text):
    keywords = kw_extractor.extract_keywords(text)
    return [x for x,y in keywords]

pandarallel.initialize(progress_bar=True)
stories['keywords'] = stories['snippet'].parallel_apply(get_keywords)

# display
stories

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet,keywords
0,Man suspected of assassination attempt against...,2024-09-23,2024-09-24 01:06:28+00:00,en,cnn.com,https://www.cnn.com/2024/09/23/politics/ryan-w...,https://web.archive.org/web/20240924010628id_/...,https://web.archive.org/web/20240924010628/htt...,https://wayback-api.archive.org/colsearch/v1/m...,The man who authorities say sat with a rifle i...,"[Ryan Wesley Routh, Routh, Secret Service agen..."
1,Kamala Harris' presidential betting lead grows...,2024-09-23,2024-09-24 01:09:20+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/e...,https://web.archive.org/web/20240924010920id_/...,https://web.archive.org/web/20240924010920/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Kamala Harris pulls further ahead in president...,"[Kamala Harris, President Donald Trump, Presid..."
2,RFK Jr. asks Supreme Court to keep his name on...,2024-09-23,2024-09-24 01:15:44+00:00,en,cnn.com,https://www.cnn.com/2024/09/23/politics/rfk-jr...,https://web.archive.org/web/20240924011544id_/...,https://web.archive.org/web/20240924011544/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Robert F. Kennedy Jr. asked the Supreme Court ...,"[President Donald Trump, Monday toÂ, backed fo..."
3,Nebraska change that could hand Donald Trump w...,2024-09-23,2024-09-24 01:12:31+00:00,en,newsweek.com,https://www.newsweek.com/nebraska-change-votes...,https://web.archive.org/web/20240924011231id_/...,https://web.archive.org/web/20240924011231/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Republican South Carolina Senator Lindsey Grah...,"[Electoral College votes, Electoral College, R..."
4,Judge detains Ryan Routh in Trump stakeout 'as...,2024-09-23,2024-09-24 01:10:27+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/2...,https://web.archive.org/web/20240924011027id_/...,https://web.archive.org/web/20240924011027/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Judge detains Ryan Routh after prosecutors say...,"[Routh, Ryan Routh, detains Ryan Routh, Trump ..."
...,...,...,...,...,...,...,...,...,...,...,...
4230,Kamala Harris’s Main-Character Energy,2024-08-23,2024-08-24 01:58:41+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/arts/televi...,https://web.archive.org/web/20240824015841id_/...,https://web.archive.org/web/20240824015841/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Supported by\nCritic’s Notebook\nKamala Harris...,"[Trump, Harris, President Donald Trump, Conven..."
4231,"‘Joy,’ ‘Freedom,’ ‘Goldilocks’: Kamala Harris’...",2024-08-23,2024-08-24 02:08:37+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/books/revie...,https://web.archive.org/web/20240824020837id_/...,https://web.archive.org/web/20240824020837/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"Supported by\nCritic’s Notebook\n‘Joy,’ ‘Freed...","[Kamala Harris, Democratic Party, Democratic N..."
4232,Why Kamala Harris’s Centrism Is Working,2024-08-23,2024-08-24 01:52:03+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/briefing/ka...,https://web.archive.org/web/20240824015203id_/...,https://web.archive.org/web/20240824015203/htt...,https://wayback-api.archive.org/colsearch/v1/m...,The Morning\nKamala Harris capped her first mo...,"[Harris, Democratic, Trump, Democratic Party, ..."
4233,Kamala Harris Wants to Win,2024-08-23,2024-08-24 02:11:07+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/opinion/ezr...,https://web.archive.org/web/20240824021107id_/...,https://web.archive.org/web/20240824021107/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"On Thursday night, Kamala Harris reintroduced ...","[Kamala Harris reintroduced, Thursday night, K..."


## Embeddings

In [9]:
import os
import openai
import dotenv
dotenv.load_dotenv()

openai.organization = None
openai.api_key = os.getenv("OPENAI_API_KEY")
# openai.Model.list() # see all openai models

In [10]:
# Import modules
import tiktoken
from openai import OpenAI
client = OpenAI()

# Set embedding model parameters
embedding_model = "text-embedding-3-small" # this is the model we will use to make embeddings
embedding_encoding = "cl100k_base"  # this the encoding for text-embedding-ada-002
max_tokens = 8000  # the maximum for text-embedding-ada-002 is 8191

# Get the encoding for the specified model
encoding = tiktoken.get_encoding(embedding_encoding)

# Make a new column with the combined title and summary
stories["combined"] = (
    "Title: " + stories.title.str.strip() + "; Content: " + stories.snippet.str.strip()
)

# Make a new column with the number of tokens in the combined title and summary
stories["n_tokens"] = stories.combined.apply(lambda x: len(encoding.encode(x)))

# Sort by that column
stories = stories.sort_values(by='n_tokens', ascending=False)

# Display the bills
stories


,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet,keywords,combined,n_tokens
1813,What was said during Harris-Trump presidential...,2024-09-11,2024-09-12 01:01:33+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/e...,https://web.archive.org/web/20240912010133id_/...,https://web.archive.org/web/20240912010133/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Read the full transcript of ABC News' presiden...,"[President Donald Trump, President Kamala Harr...",Title: What was said during Harris-Trump presi...,19398
15,"Live updates: Donald Trump, Kamala Harris elec...",2024-09-23,2024-09-24 01:14:00+00:00,en,cnn.com,https://www.cnn.com/politics/live-news/trump-h...,https://web.archive.org/web/20240924011400id_/...,https://web.archive.org/web/20240924011400/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Nebraska setback for Trump: A pressure campaig...,"[President Donald Trump, President Kamala Harr...","Title: Live updates: Donald Trump, Kamala Harr...",14067
2366,Where Kamala Harris and Donald Trump Stand on ...,2024-09-09,2024-09-11 01:50:52+00:00,en,nytimes.com,https://www.nytimes.com/interactive/2024/us/po...,https://web.archive.org/web/20240911015052id_/...,https://web.archive.org/web/20240911015052/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Where Kamala Harris and Donald Trump Stand on ...,"[Trump, Inflation Reduction Act, United States...",Title: Where Kamala Harris and Donald Trump St...,13928
2820,Harris and Trump Settle on Debate Rules: 2024 ...,2024-09-04,2024-09-05 01:27:25+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/04/us/har...,https://web.archive.org/web/20240905012725id_/...,https://web.archive.org/web/20240905012725/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Harris and Trump Settle...,"[President Kamala Harris, Vice President Kamal...",Title: Harris and Trump Settle on Debate Rules...,13278
2239,Harris and Trump square off in their first pre...,2024-09-10,2024-09-12 01:31:30+00:00,en,politico.com,https://www.politico.com/live-updates/2024/09/...,https://web.archive.org/web/20240912013130id_/...,https://web.archive.org/web/20240912013130/htt...,https://wayback-api.archive.org/colsearch/v1/m...,A majority of voters who watched Tuesday’s deb...,"[Donald Trump, Kamala Harris, President Kamala...",Title: Harris and Trump square off in their fi...,12376
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1025,Assassination attempt on Trump investigated,2024-09-16,2024-09-18 01:32:51+00:00,en,cbsnews.com,https://www.cbsnews.com/miami/video/assassinat...,https://web.archive.org/web/20240918013251id_/...,https://web.archive.org/web/20240918013251/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Assassination attempt on Trump investigated\nC...,"[President Donald Trump, Miami Joan Murray, Tr...",Title: Assassination attempt on Trump investig...,59
1914,Reflecting on first debate between Harris and ...,2024-09-11,2024-09-13 01:47:20+00:00,en,cbsnews.com,https://www.cbsnews.com/video/reflecting-on-fi...,https://web.archive.org/web/20240913014720id_/...,https://web.archive.org/web/20240913014720/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Reflecting on first debate between Harris and ...,"[President Donald Trump, President Kamala Harr...",Title: Reflecting on first debate between Harr...,59
231,Secret Service chief details failures during T...,2024-09-20,2024-09-21 02:11:21+00:00,en,cbsnews.com,https://www.cbsnews.com/video/secret-service-c...,https://web.archive.org/web/20240921021121id_/...,https://web.archive.org/web/20240921021121/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Secret Service chief details failures during T...,"[Service chief details, Secret Service chief, ...",Title: Secret Service chief details failures d...,59
1934,Pennsylvania voters react to Harris-Trump debate,2024-09-10,2024-09-12 01:57:33+00:00,en,

In [11]:
# Grab the rows where the text is too big for the context window of the mmodel (>8000 tokens)
too_long = stories.query("n_tokens > @max_tokens") 

# Print how many will be removed
print(f"Removing {len(too_long)} stories that are too long")

# Display the removed stories here in this cell so we can see what we're losing
display(too_long)  

# Remove the rows where the text is too big for the context window of the model
stories = stories.query("n_tokens <= @max_tokens")  

Removing 28 stories that are too long


,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet,keywords,combined,n_tokens
1813,What was said during Harris-Trump presidential...,2024-09-11,2024-09-12 01:01:33+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/e...,https://web.archive.org/web/20240912010133id_/...,https://web.archive.org/web/20240912010133/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Read the full transcript of ABC News' presiden...,"[President Donald Trump, President Kamala Harr...",Title: What was said during Harris-Trump presi...,19398
15,"Live updates: Donald Trump, Kamala Harris elec...",2024-09-23,2024-09-24 01:14:00+00:00,en,cnn.com,https://www.cnn.com/politics/live-news/trump-h...,https://web.archive.org/web/20240924011400id_/...,https://web.archive.org/web/20240924011400/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Nebraska setback for Trump: A pressure campaig...,"[President Donald Trump, President Kamala Harr...","Title: Live updates: Donald Trump, Kamala Harr...",14067
2366,Where Kamala Harris and Donald Trump Stand on ...,2024-09-09,2024-09-11 01:50:52+00:00,en,nytimes.com,https://www.nytimes.com/interactive/2024/us/po...,https://web.archive.org/web/20240911015052id_/...,https://web.archive.org/web/20240911015052/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Where Kamala Harris and Donald Trump Stand on ...,"[Trump, Inflation Reduction Act, United States...",Title: Where Kamala Harris and Donald Trump St...,13928
2820,Harris and Trump Settle on Debate Rules: 2024 ...,2024-09-04,2024-09-05 01:27:25+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/04/us/har...,https://web.archive.org/web/20240905012725id_/...,https://web.archive.org/web/20240905012725/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Harris and Trump Settle...,"[President Kamala Harris, Vice President Kamal...",Title: Harris and Trump Settle on Debate Rules...,13278
2239,Harris and Trump square off in their first pre...,2024-09-10,2024-09-12 01:31:30+00:00,en,politico.com,https://www.politico.com/live-updates/2024/09/...,https://web.archive.org/web/20240912013130id_/...,https://web.archive.org/web/20240912013130/htt...,https://wayback-api.archive.org/colsearch/v1/m...,A majority of voters who watched Tuesday’s deb...,"[Donald Trump, Kamala Harris, President Kamala...",Title: Harris and Trump square off in their fi...,12376
1792,2024 Election Live Updates: Latest Trump and H...,2024-09-11,2024-09-12 01:12:21+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/11/us/har...,https://web.archive.org/web/20240912011221id_/...,https://web.archive.org/web/20240912011221/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Debate Ripples Across C...,"[President Kamala Harris, Trump, Vice Presiden...",Title: 2024 Election Live Updates: Latest Trum...,11494
2716,Trump and Harris Campaign News: 2024 Election ...,2024-09-05,2024-09-06 01:34:27+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/05/us/har...,https://web.archive.org/web/20240906013427id_/...,https://web.archive.org/web/20240906013427/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Vance Rallies Supporter...,"[President Kamala Harris, Trump, Vice Presiden...",Title: Trump and Harris Campaign News: 2024 El...,11167
3771,Election Live Updates: Trump Suggests Debate R...,2024-08-27,2024-08-28 02:13:29+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/08/27/us/har...,https://web.archive.org/web/20240828021329id_/...,https://web.archive.org/web/20240828021329/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Trump Suggests Debate R...,"[President Kamala Harris, Vice President Kamal...",Title: Election Live Updates: Trump Suggests D...,11145
847,Apparent Trump Assassination Attempt and Suspe...,2024-09-17,2024-09-18 01:07:17+00:00,en,nytimes.com,https://www.nytimes.com/live

In [12]:
# define a function to get embeeddings
def get_embedding(text, model="text-embedding-3-small"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input = [text], model=model).data[0].embedding

# Get the embeddings for the combined title and summary
stories['embedding'] = stories.combined.progress_apply(lambda x: get_embedding(x, model='text-embedding-3-small'))

# Save to CSV
stories.to_csv('stories-with-embeddings.csv')

# Display the dataframe with embeddings
stories.head(2)

  0%|          | 0/4061 [00:00<?, ?it/s]

,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet,keywords,combined,n_tokens,embedding
2313,"Trump, Harris lay low ahead of debate as wildf...",2024-09-09,2024-09-10 02:04:09+00:00,en,newsweek.com,https://www.newsweek.com/election-2024-kamala-...,https://web.archive.org/web/20240910020409id_/...,https://web.archive.org/web/20240910020409/htt...,https://wayback-api.archive.org/colsearch/v1/m...,As the anticipation builds for the presidentia...,"[President Kamala Harris, Vice President Kamal...","Title: Trump, Harris lay low ahead of debate a...",8000,"[0.029513994231820107, 0.019624708220362663, 0..."
4218,Transcript: Ezra Klein on Kamala Harris’s Conv...,2024-08-23,2024-08-25 01:54:08+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/podcasts/tr...,https://web.archive.org/web/20240825015408id_/...,https://web.archive.org/web/20240825015408/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Supported by\nThe Ezra Klein Show\nTranscript:...,"[Ezra Klein, Aaron Retica, Donald Trump, Kamal...",Title: Transcript: Ezra Klein on Kamala Harris...,8000,"[0.04471578821539879, 0.0195614006370306, -0.0..."


In [13]:
# drop combined column since those were only for the purposes of making the embeddings
stories = stories.drop(columns=['combined', 'n_tokens'])

## Dimensionality Reduction (t-SNE)


In [14]:
from sklearn.manifold import TSNE
import numpy as np

# check if vis_dims exists
if os.path.exists("data/stories-with-vis-dims.csv"):
    stories = pd.read_csv("data/stories-with-vis-dims.csv")
else: 
    # Convert to a list of lists of floats
    matrix = np.array(stories.embedding.to_list())

    # Create a t-SNE model and transform the data
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='random', learning_rate=400)
    vis_dims = tsne.fit_transform(matrix)

    # add to dataframe and write to csv
    stories = stories\
        .assign(
            x = vis_dims[:,0], 
            y = vis_dims[:,1])


In [15]:
stories.to_csv('output/stories-with-nlp.csv', index=False)
stories.head()

,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet,keywords,embedding,x,y
2313,"Trump, Harris lay low ahead of debate as wildf...",2024-09-09,2024-09-10 02:04:09+00:00,en,newsweek.com,https://www.newsweek.com/election-2024-kamala-...,https://web.archive.org/web/20240910020409id_/...,https://web.archive.org/web/20240910020409/htt...,https://wayback-api.archive.org/colsearch/v1/m...,As the anticipation builds for the presidentia...,"[President Kamala Harris, Vice President Kamal...","[0.029513994231820107, 0.019624708220362663, 0...",0.702381,-12.091496
4218,Transcript: Ezra Klein on Kamala Harris’s Conv...,2024-08-23,2024-08-25 01:54:08+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/podcasts/tr...,https://web.archive.org/web/20240825015408id_/...,https://web.archive.org/web/20240825015408/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Supported by\nThe Ezra Klein Show\nTranscript:...,"[Ezra Klein, Aaron Retica, Donald Trump, Kamal...","[0.04471578821539879, 0.0195614006370306, -0.0...",-12.611116,11.818191
1976,"Harris, Trump trade barbs in heated, high-stak...",2024-09-10,2024-09-12 01:03:31+00:00,en,latimes.com,https://www.latimes.com/politics/story/2024-09...,https://web.archive.org/web/20240912010331id_/...,https://web.archive.org/web/20240912010331/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"Harris, Trump trade barbs in heated, high-stak...","[President Kamala Harris, Vice President Kamal...","[-0.00299630593508482, 0.0061648134142160416, ...",25.161629,-1.686306
56,Kamala Harris Gets National Security Endorseme...,2024-09-22,2024-09-23 01:05:32+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/22/us/tru...,https://web.archive.org/web/20240923010532id_/...,https://web.archive.org/web/20240923010532/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Mark Robinson’s Top Aid...,"[President Donald Trump, President Kamala Harr...","[0.04099341481924057, 0.03667009621858597, 0.0...",-17.810658,-8.052394
2168,"Debate fact check: What Harris, Trump got wron...",2024-09-10,2024-09-12 01:03:50+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/e...,https://web.archive.org/web/20240912010350id_/...,https://web.archive.org/web/20240912010350/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Presidential debate fact check: Analyzing Trum...,"[Donald Trump claim, Kamala Harris claim, Dona...","[0.0005179689615033567, 0.02802026830613613, 0...",25.699959,8.026423


# Topic Modeling

In [16]:
stories.reset_index(drop=True, inplace=True)

In [17]:
stories

,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet,keywords,embedding,x,y
0,"Trump, Harris lay low ahead of debate as wildf...",2024-09-09,2024-09-10 02:04:09+00:00,en,newsweek.com,https://www.newsweek.com/election-2024-kamala-...,https://web.archive.org/web/20240910020409id_/...,https://web.archive.org/web/20240910020409/htt...,https://wayback-api.archive.org/colsearch/v1/m...,As the anticipation builds for the presidentia...,"[President Kamala Harris, Vice President Kamal...","[0.029513994231820107, 0.019624708220362663, 0...",0.702381,-12.091496
1,Transcript: Ezra Klein on Kamala Harris’s Conv...,2024-08-23,2024-08-25 01:54:08+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/podcasts/tr...,https://web.archive.org/web/20240825015408id_/...,https://web.archive.org/web/20240825015408/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Supported by\nThe Ezra Klein Show\nTranscript:...,"[Ezra Klein, Aaron Retica, Donald Trump, Kamal...","[0.04471578821539879, 0.0195614006370306, -0.0...",-12.611116,11.818191
2,"Harris, Trump trade barbs in heated, high-stak...",2024-09-10,2024-09-12 01:03:31+00:00,en,latimes.com,https://www.latimes.com/politics/story/2024-09...,https://web.archive.org/web/20240912010331id_/...,https://web.archive.org/web/20240912010331/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"Harris, Trump trade barbs in heated, high-stak...","[President Kamala Harris, Vice President Kamal...","[-0.00299630593508482, 0.0061648134142160416, ...",25.161629,-1.686306
3,Kamala Harris Gets National Security Endorseme...,2024-09-22,2024-09-23 01:05:32+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/22/us/tru...,https://web.archive.org/web/20240923010532id_/...,https://web.archive.org/web/20240923010532/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Mark Robinson’s Top Aid...,"[President Donald Trump, President Kamala Harr...","[0.04099341481924057, 0.03667009621858597, 0.0...",-17.810658,-8.052394
4,"Debate fact check: What Harris, Trump got wron...",2024-09-10,2024-09-12 01:03:50+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/e...,https://web.archive.org/web/20240912010350id_/...,https://web.archive.org/web/20240912010350/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Presidential debate fact check: Analyzing Trum...,"[Donald Trump claim, Kamala Harris claim, Dona...","[0.0005179689615033567, 0.02802026830613613, 0...",25.699959,8.026423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4056,Assassination attempt on Trump investigated,2024-09-16,2024-09-18 01:32:51+00:00,en,cbsnews.com,https://www.cbsnews.com/miami/video/assassinat...,https://web.archive.org/web/20240918013251id_/...,https://web.archive.org/web/20240918013251/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Assassination attempt on Trump investigated\nC...,"[President Donald Trump, Miami Joan Murray, Tr...","[0.024831069633364677, 0.04886317253112793, -0...",14.755981,55.705868
4057,Reflecting on first debate between Harris and ...,2024-09-11,2024-09-13 01:47:20+00:00,en,cbsnews.com,https://www.cbsnews.com/video/reflecting-on-fi...,https://web.archive.org/web/20240913014720id_/...,https://web.archive.org/web/20240913014720/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Reflecting on first debate between Harris and ...,"[President Donald Trump, President Kamala Harr...","[-0.010606903582811356, -0.012452526949346066,...",48.667713,-20.478518
4058,Secret Service chief details failures during T...,2024-09-20,2024-09-21 02:11:21+00:00,en,cbsnews.com,https://www.cbsnews.com/video/secret-service-c...,https://web.archive.org/web/20240921021121id_/...,https://web.archive.org/web/20240921021121/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Secret Service chief details failures during T...,"[Service chief details, Secret Service chief, ...","[-0.00765483221039176, 0.02017

In [18]:
from sklearn.cluster import DBSCAN
# Convert embedding to a NumPy array
X = np.stack(stories['embedding'].values)

# Perform DBSCAN clustering
dbscan = DBSCAN(eps=0.5, min_samples=10)  # Adjust eps and min_samples as per your requirement
labels = dbscan.fit_predict(X)

# Assign topics to DataFrame
stories['topic'] = labels

# Group articles by topic
grouped = stories.groupby('topic')

# sort groups by size
grouped = sorted(grouped, key=lambda x: len(x[1]), reverse=True)

# assign group numbers back to stories
for i, (name, group) in enumerate(grouped):
    # TODO: I THINK THIS IS BROKEN 🐛, getting weird items into
    stories.loc[stories['topic'] == name, 'topic'] = name

print("Number of groups:", len(grouped))
# Number of items in each group
print("Group sizes:")
print([len(group) for name, group in grouped])



Number of groups: 14
Group sizes:
[3543, 265, 57, 36, 34, 22, 20, 17, 15, 11, 11, 10, 10, 10]


In [19]:
def summarize_topic(titles):
    """
    Pass list of titles to ChatGPT and ask it to summarize them in 2-4 words.
    """

    # Combine the titles into a single string
    titles_str = ', '.join(titles)

    # print("Writing a title for")
    # for title in titles[:5]:
    #     print(f"  - {title}")
    
    MODEL = "gpt-3.5-turbo"
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"The following article titles form a topic. \n\n {titles_str} \n\n Please write a specific summary of the topic in 2-4 words:"},
        ],
        max_tokens=10
    )

    return response.choices[0].message.content

In [20]:
# make a list of titles per topic
topic_titles = stories.groupby('topic')['title'].apply(list).to_dict()
topic_titles = [{
    'topic': k,
    'num_articles': len(v),
    'headlines': v
} for k,v in topic_titles.items()]

# sort by num_articles
topic_titles = sorted(topic_titles, key=lambda x: x['num_articles'], reverse=True)

# pass each topic list of titles to openai chatgpt and ask it to summarize the topic in 2-4 words
for topic in topic_titles[0:]:
    print(f"Topic {topic['topic']} ({topic['num_articles']} articles)")
    
    if topic['topic'] == -1:
        continue

    # if there are more than 10 articles in a topic, sample 10 (to keep within the word limit of the API)
    if topic['num_articles'] >= 10:
        headlines = np.random.choice(topic['headlines'], 10, replace=False)
        # Summary
        try:
            topic['topic_summary'] = summarize_topic(headlines)
            print(topic['topic_summary'])
        except InvalidRequestError:
            topic['topic_summary'] = "Error Making Summary From OpenAI API"
            print("OpenAI API request failed.")
    else:
        headlines = topic['headlines']

Topic -1 (3543 articles)
Topic 0 (265 articles)
Trump-Harris Debate Overview.
Topic 2 (57 articles)
Trump assassination attempt coverage.
Topic 9 (36 articles)
Trump-Harris Debate Analysis
Topic 1 (34 articles)
Taylor Swift's Political Endorsements
Topic 3 (22 articles)
"Trump Incident at Arlington"
Topic 7 (20 articles)
Republicans endorse Kamala Harris.
Topic 10 (17 articles)
Trump Election Subversion Hearing
Topic 11 (15 articles)
Trump Media stock fluctuations
Topic 5 (11 articles)
Trump's hush money sentencing battle.
Topic 12 (11 articles)
Harris fundraising outpaces Trump.
Topic 4 (10 articles)
Trump Election Case Developments
Topic 6 (10 articles)
RFK Jr. endorses Trump
Topic 8 (10 articles)
Jack Smith's Trump Indictment


In [21]:
# turn topic titles and summaries into a dataframe
topic_titles_df = pd.DataFrame(topic_titles)
topic_titles_df = topic_titles_df[['topic', 'topic_summary']]
stories = stories.merge(topic_titles_df, on='topic', how='left')

In [22]:
stories.to_csv('../stories-with-embeddings.csv', index=False)


# Collect Metadata

In [23]:
# loop through topic_titles
topic_metadata = []
for topic in topic_titles:
    # grab topic, num_articles, and summary only
    topic = {k:v for k,v in topic.items() if k in ['topic', 'num_articles', 'topic_summary']}
    topic_metadata.append(topic)

topic_metadata

[{'topic': -1, 'num_articles': 3543},
 {'topic': 0,
  'num_articles': 265,
  'topic_summary': 'Trump-Harris Debate Overview.'},
 {'topic': 2,
  'num_articles': 57,
  'topic_summary': 'Trump assassination attempt coverage.'},
 {'topic': 9,
  'num_articles': 36,
  'topic_summary': 'Trump-Harris Debate Analysis'},
 {'topic': 1,
  'num_articles': 34,
  'topic_summary': "Taylor Swift's Political Endorsements"},
 {'topic': 3,
  'num_articles': 22,
  'topic_summary': '"Trump Incident at Arlington"'},
 {'topic': 7,
  'num_articles': 20,
  'topic_summary': 'Republicans endorse Kamala Harris.'},
 {'topic': 10,
  'num_articles': 17,
  'topic_summary': 'Trump Election Subversion Hearing'},
 {'topic': 11,
  'num_articles': 15,
  'topic_summary': 'Trump Media stock fluctuations'},
 {'topic': 5,
  'num_articles': 11,
  'topic_summary': "Trump's hush money sentencing battle."},
 {'topic': 12,
  'num_articles': 11,
  'topic_summary': 'Harris fundraising outpaces Trump.'},
 {'topic': 4,
  'num_articles'

In [24]:
# read output/metadata.json
import json
with open('output/metadata.json') as f:
    metadata = json.load(f)

metadata['topics'] = topic_metadata

# write metadata back to json file
with open('output/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

metadata

{'start': '2024-08-23T00:00:00',
 'end': '2024-09-25T21:18:55.601423',
 'query': '(title:Kamala OR title:Trump)',
 'query_raw': '(title:Kamala OR title:Trump) AND language:en AND domain:(nytimes.com OR cnn.com OR foxnews.com OR nypost.com OR washingtonpost.com OR usatoday.com OR cnbc.com OR theguardian.com OR breakingnews.com OR buzzfeed.com OR cbsnews.com OR reuters.com OR huffingtonpost.com OR usnews.com OR latimes.com OR politico.com OR newsweek.com OR breitbart.com)',
 'topics': [{'topic': -1, 'num_articles': 3543},
  {'topic': 0,
   'num_articles': 265,
   'topic_summary': 'Trump-Harris Debate Overview.'},
  {'topic': 2,
   'num_articles': 57,
   'topic_summary': 'Trump assassination attempt coverage.'},
  {'topic': 9,
   'num_articles': 36,
   'topic_summary': 'Trump-Harris Debate Analysis'},
  {'topic': 1,
   'num_articles': 34,
   'topic_summary': "Taylor Swift's Political Endorsements"},
  {'topic': 3,
   'num_articles': 22,
   'topic_summary': '"Trump Incident at Arlington"'}

In [25]:
# collect top keywords
top_keywords = stories\
    .explode('keywords')\
    .groupby('keywords')\
    .size()\
    .reset_index(name='count')\
    .sort_values(by='count', ascending=False)\
    .head(100)
    

In [26]:
stories.query("title.str.contains('Harris') & ~title.str.contains('Trump')")

,title,publication_date,capture_time,language,domain,url,original_capture_url,archive_playback_url,article_url,snippet,keywords,embedding,x,y,topic,topic_summary
1,Transcript: Ezra Klein on Kamala Harris’s Conv...,2024-08-23,2024-08-25 01:54:08+00:00,en,nytimes.com,https://www.nytimes.com/2024/08/23/podcasts/tr...,https://web.archive.org/web/20240825015408id_/...,https://web.archive.org/web/20240825015408/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Supported by\nThe Ezra Klein Show\nTranscript:...,"[Ezra Klein, Aaron Retica, Donald Trump, Kamal...","[0.04471578821539879, 0.0195614006370306, -0.0...",-12.611116,11.818191,-1,NaN
3,Kamala Harris Gets National Security Endorseme...,2024-09-22,2024-09-23 01:05:32+00:00,en,nytimes.com,https://www.nytimes.com/live/2024/09/22/us/tru...,https://web.archive.org/web/20240923010532id_/...,https://web.archive.org/web/20240923010532/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Election Live Updates: Mark Robinson’s Top Aid...,"[President Donald Trump, President Kamala Harr...","[0.04099341481924057, 0.03667009621858597, 0.0...",-17.810658,-8.052394,-1,NaN
11,The Kamala Harris and Tim Walz CNN interview w...,2024-08-29,2024-08-31 01:24:04+00:00,en,usatoday.com,https://www.usatoday.com/story/news/politics/e...,https://web.archive.org/web/20240831012404id_/...,https://web.archive.org/web/20240831012404/htt...,https://wayback-api.archive.org/colsearch/v1/m...,As Harris-Walz interview captures nation’s att...,"[Bash, HARRIS, Vice President, American people...","[0.0614611878991127, -0.028726425021886826, -0...",0.142859,-26.414055,-1,NaN
13,‘I think I am breaking the pattern’: Janet Jac...,2024-09-21,2024-09-22 01:54:31+00:00,en,theguardian.com,https://www.theguardian.com/music/2024/sep/21/...,https://web.archive.org/web/20240922015431id_/...,https://web.archive.org/web/20240922015431/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Janet Jackson has a cold. It is the hottest da...,"[Jackson, n’t, Janet Jackson, Janet Jackson so...","[0.05902419984340668, 0.001965947914868593, 0....",-23.571739,18.780479,-1,NaN
14,Kamala Harris's DNC acceptance speech—fact-che...,2024-08-23,2024-08-24 01:58:03+00:00,en,newsweek.com,https://www.newsweek.com/kamala-harris-accepta...,https://web.archive.org/web/20240824015803id_/...,https://web.archive.org/web/20240824015803/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Kamala Harris delivered her acceptance speech ...,"[Trump, Donald Trump, Chicago on Thursday, Har...","[-0.01930306665599346, 0.02832595445215702, 0....",28.452803,28.901793,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3986,"President Joe Biden, Vice President Kamala Har...",2024-09-02,2024-09-04 01:49:27+00:00,en,cbsnews.com,https://www.cbsnews.com/pittsburgh/video/presi...,https://web.archive.org/web/20240904014927id_/...,https://web.archive.org/web/20240904014927/htt...,https://wayback-api.archive.org/colsearch/v1/m...,"President Joe Biden, Vice President Kamala Har...","[Vice President Kamala, President Joe Biden, P...","[0.013509190641343594, -0.03034704551100731, 0...",49.827412,-38.249680,-1,NaN
4016,"Kamala Harris DNC speech strategy, fact checki...",2024-08-23,2024-08-24 02:09:50+00:00,en,cbsnews.com,https://www.cbsnews.com/video/kamala-harris-dn...,https://web.archive.org/web/20240824020950id_/...,https://web.archive.org/web/20240824020950/htt...,https://wayback-api.archive.org/colsearch/v1/m...,Vice President Kamala Harris worked on her spe...,"[President Kamala Harris, Democratic National ...","[0.0542781837284565, 0.004729287698864937, 0.0...",3.680733,-44.934013,-1,NaN
4020,What Kamala Harris told Latinos at Congression...,2024-09-18,2024-09-19 01:26:16+00:00,en,cbsnews.com,https://www.cbsnews.com/video/what-kamala-harr...,https://web.archive.org/web/20240919012616id_/...,https://web.archive.org/web/20240919012616/htt...,https://wayback-api.archive.org/colsearch/v1/m...,What Kamala Harris told Latinos at Congress